With Pipeline

The complete original notebook 04 (`../archive/notebooks/04_eda_and_multivariate_main.ipynb`): EDA, all models
(Linear regression, LightGBM, Random Forest, XGBoost, Polynomial Ridge), error analysis
and saving the model.

Data, features, splits and models all come from the pipeline:

* `modeling/hydro_pipeline.py` → daily data, features, split, metrics
* `modeling/kaub_pipeline.py` → models, training, saving, forecasting

In [ ]:
# importing libraries
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 140)

In [ ]:
# importing the pipelines
from modeling import hydro_pipeline as hp
from modeling import kaub_pipeline as kp

H = kp.HORIZON                          # 7 days ahead
LOW_WATER_THRESHOLD = hp.LOW_WATER_CM   # 77 cm

## 1. Data

In [ ]:
# Check duplicate timestamps in the raw files (the 02:00 hour repeats every October)
dup_check = {}

for name, path in hp.GAUGES.items():
    d = pd.read_csv(path, sep=";")
    d["timestamp"] = pd.to_datetime(d["timestamp"])

    dups = d.loc[d["timestamp"].duplicated(keep=False), "timestamp"]

    dup_check[name] = {
        "n duplicated rows": len(dups),
        "distinct dates affected": dups.dt.date.nunique(),
        "months affected": sorted(dups.dt.month.unique().tolist()),
        "hours affected": sorted(dups.dt.hour.unique().tolist()),
    }

pd.DataFrame(dup_check).T

In [ ]:
# Load the daily hydro-weather dataset with the pipeline
df = kp.load_data()

print(df.shape)
df.head()

In [ ]:
# Check complete days: compare hours_available with hours_expected (23, 24 or 25)
df[["hours_expected", "hours_available"]].value_counts().sort_index()

## 2. Exploratory analysis

In [ ]:
# Select the main hydrological variables for correlation analysis
cols = [
    "water_level_mean",
    "bingen_level_mean",
    "discharge_mean"
]

df[cols].corr()

In [ ]:
# Heatmap of the hydrological variables
plt.figure(figsize=(7, 5))

sns.heatmap(df[cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)

plt.title("Correlation of Hydrological Variables")
plt.show()

In [ ]:
# Correlation between Kaub and lagged Bingen water levels (0-7 days)
for lag in range(0, 8):
    corr = df["water_level_mean"].corr(df["bingen_level_mean"].shift(lag))
    print(f"Bingen lag {lag}: r = {corr:.3f}")

In [ ]:
# Correlation between Kaub and lagged discharge (0-7 days)
for lag in range(0, 8):
    corr = df["water_level_mean"].corr(df["discharge_mean"].shift(lag))
    print(f"Discharge lag {lag}: r = {corr:.3f}")

In [ ]:
# Create all features and targets with the pipeline
features = kp.make_features(df)

df = df.join(features)

print(features.columns.tolist())

In [ ]:
# Which rainfall window is most related to the Kaub water level?
rain_cols = [
    "precipitation_sum",
    "precip_3d",
    "precip_7d",
    "precip_14d",
    "precip_30d"
]

df[rain_cols + ["water_level_mean"]].corr()["water_level_mean"]

In [ ]:
# Correlation matrix of hydrological and weather variables
multi_cols = [
    "water_level_mean", "temperature_mean", "humidity_mean", "precipitation_sum",
    "pressure_mean", "wind_speed_mean", "bingen_level_mean", "discharge_mean",
    "precip_3d", "precip_7d", "precip_14d", "precip_30d"
]

plt.figure(figsize=(11, 8))
sns.heatmap(df[multi_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Multivariate Correlation Matrix")
plt.show()

### Seasonal pattern across the year

Every year since 2000 on a January–December axis, with the typical range
(10th–90th percentile) and the median.

In [ ]:
# Typical year: median and 10th-90th percentile for each day of the year
level = df["water_level_mean"]
doy = level.index.dayofyear

clim = level.groupby(doy).quantile([0.1, 0.5, 0.9]).unstack()
clim = clim.rolling(15, center=True, min_periods=1).mean()

last_years = sorted(level.index.year.unique())[-2:]
year_colors = {last_years[0]: "#eb6834", last_years[1]: "#1baf7a"}

In [ ]:
# Plot every year (grey) and the last two years (colour)
fig, ax = plt.subplots(figsize=(13, 5))

for year, s in level.groupby(level.index.year):
    if year not in last_years:
        ax.plot(s.index.dayofyear, s.values, color="grey", lw=0.5, alpha=0.25)

ax.fill_between(clim.index, clim[0.1], clim[0.9], color="#2a78d6", alpha=0.2,
                label="10th–90th percentile")
ax.plot(clim.index, clim[0.5], color="#1c5cab", lw=2, label="Median")

for year in last_years:
    s = level[level.index.year == year]
    ax.plot(s.index.dayofyear, s.values, color=year_colors[year], lw=2, label=str(year))

months = pd.date_range("2001-01-01", periods=12, freq="MS")
ax.set_xticks(months.dayofyear, months.strftime("%b"))
ax.set_ylabel("Water level (cm)")
ax.set_title("Kaub water level across the year")
ax.legend(ncol=2)
plt.show()

In [ ]:
# Zoom: the last two years compared with the typical year
zoom = level[level.index.year >= last_years[0]].dropna()
zoom_doy = zoom.index.dayofyear

fig, ax = plt.subplots(figsize=(13, 4))

ax.fill_between(zoom.index, clim[0.1].reindex(zoom_doy), clim[0.9].reindex(zoom_doy),
                color="#2a78d6", alpha=0.2, label="Typical range")
ax.plot(zoom.index, clim[0.5].reindex(zoom_doy).to_numpy(), color="#1c5cab", ls="--",
        label="Typical (median)")

for year in last_years:
    s = zoom[zoom.index.year == year]
    ax.plot(s.index, s.values, color=year_colors[year], lw=1.8, label=str(year))

ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.set_ylabel("Water level (cm)")
ax.set_title(f"Zoom: {last_years[0]}–{last_years[1]}")
ax.legend(ncol=2)
plt.show()

## 3. Features and targets

In [ ]:
# Input features (all groups together) and targets 1-7 days ahead
FEATURES = kp.ALL_FEATURES
TARGETS = [f"target_day_{h}" for h in range(1, 8)]

print("FEATURES:", FEATURES)
print("TARGETS:", TARGETS)

In [ ]:
# Correlation between features and future water levels (7 horizons)
corr_horizons = df[FEATURES + TARGETS].corr().loc[FEATURES, TARGETS]

corr_horizons.round(3)

In [ ]:
# Heatmap: feature correlation across the 7 forecast horizons
plt.figure(figsize=(12, 10))

sns.heatmap(corr_horizons, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1)

plt.title("Feature Correlation with Kaub Water Level Across 7 Forecast Horizons")
plt.xlabel("Forecast Horizon")
plt.ylabel("Features")
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: correlation between the features
plt.figure(figsize=(16, 13))

sns.heatmap(df[FEATURES].corr(), annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1)

plt.title("Correlation Matrix of Multivariate Features")
plt.tight_layout()
plt.show()

In [ ]:
# Check that features and target line up with the correct dates
day = pd.Timestamp("2010-06-15")

today = df.loc[day, "lag_0"]
in_7_days = df.loc[day + pd.Timedelta(days=H), "lag_0"]

assert df.loc[day, "target"] == in_7_days

print(f"On {day.date()}: today = {today}, target (+{H}d) = {in_7_days}")

## 4. Train / validation / test split

In [ ]:
# Split by time with the pipeline: last 3 years = test, last 2 years of train = validation, 7-day gaps
splits = kp.split_data(features)

train = splits["train"]
test = splits["test"]
train_fit = splits["train_fit"]
validation = splits["validation"]

for name, part in splits.items():
    print(f"{name:10s}: {part.index.min().date()} → {part.index.max().date()} n={len(part)}")

In [ ]:
# Persistence baseline: the water level in 7 days = the water level today
y_val = validation["target"]
pred_baseline = validation["lag_0"]

persistence_mae = (pred_baseline - y_val).abs().mean()

baseline = hp.metrics(y_val, pred_baseline)
baseline["Model"] = "Persistence baseline"

pd.DataFrame([baseline]).round(3)

In [ ]:
# Feature groups to compare
feature_groups = kp.FEATURE_GROUPS

for name, columns in feature_groups.items():
    print(f"{name}: {len(columns)} features")

## 5. Models

In [ ]:
# Linear regression for every feature group (trained by the pipeline)
lr_results_df, lr_models = kp.train_model_groups("Linear regression", splits)

lr_results_df.round(3)

1. Modeling : LightGBM

In [ ]:
# LightGBM for every feature group
lgbm_results_df, lgbm_models = kp.train_model_groups("LightGBM", splits)

lgbm_results_df.round(3)

2. Random Forest :

In [ ]:
# Random forest for every feature group
rf_results_df, rf_models = kp.train_model_groups("Random forest", splits)

rf_results_df.round(3)

In [ ]:
# Best random forest
best_rf_group = rf_results_df["MAE"].idxmin()

print("Best feature group:", best_rf_group)
print(f"Validation MAE: {rf_results_df.loc[best_rf_group, 'MAE']:.3f} cm")

3. Modeling : XGBoost

In [ ]:
# XGBoost for every feature group
xgb_results_df, xgb_models = kp.train_model_groups("XGBoost", splits)

xgb_results_df.round(3)

In [ ]:
# Best XGBoost: train again on all training years, then predict the test years once
best_group = xgb_results_df["MAE"].idxmin()

final_xgb, test_predictions = kp.refit_and_test("XGBoost", best_group, splits)

print("Best feature group:", best_group)

In [ ]:
# Test results
y_test = test["target"]

test_result = hp.metrics(y_test, test_predictions, test["lag_0"])
test_result["Model"] = "XGBoost"

pd.DataFrame([test_result]).round(3)

In [ ]:
# Error table for the test data
errors = pd.DataFrame({
    "Actual": y_test,
    "Predicted": test_predictions,
    "Persistence": test["lag_0"]
})

# Positive error = overprediction
errors["Error"] = errors["Predicted"] - errors["Actual"]
errors["Absolute error"] = errors["Error"].abs()

errors.head()

In [ ]:
# Error by water condition (low / normal / high), based on the actual level
high_threshold = train["target"].quantile(0.90)

error_summary = hp.regime_table(
    errors["Actual"], errors["Predicted"], errors["Persistence"],
    low_cm=LOW_WATER_THRESHOLD, high_cm=high_threshold
)

print(f"Low water < {LOW_WATER_THRESHOLD} cm, high water >= {high_threshold:.0f} cm")
error_summary.round(2)

In [ ]:
# Bar chart: XGBoost vs persistence per water condition
plot_data = error_summary.loc[["Low water", "Normal water", "High water"],
                              ["MAE", "Persistence MAE"]]
plot_data.columns = ["XGBoost", "Persistence"]

ax = plot_data.plot.bar(figsize=(9, 5), color=["#2a78d6", "#eb6834"], rot=0, width=0.7)

for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", padding=3)

ax.set(title="Test Error by Water Condition", xlabel="Water condition",
       ylabel="MAE (cm) — lower is better")
plt.tight_layout()
plt.show()

In [ ]:
# Low-water alarm: will Kaub be below the threshold in 7 days? (watch Recall: missed alarms are costly)
pd.DataFrame({
    "XGBoost": hp.alarm_scores(y_test, test_predictions),
    "Persistence": hp.alarm_scores(y_test, test["lag_0"]),
}).T.round(3)

In [ ]:
# The 10 biggest test errors
errors["Target date"] = (errors.index + pd.Timedelta(days=H)).date

errors.nlargest(10, "Absolute error")[
    ["Target date", "Actual", "Predicted", "Error", "Absolute error"]
].round(2)

In [ ]:
# Over-fitting check: training vs validation error
group = "Kaub + precipitation + temperature"
model = xgb_models[group]
columns = feature_groups[group]

comparison = []

for name, subset in [("Training", train_fit), ("Validation", validation)]:
    row = hp.metrics(subset["target"], model.predict(subset[columns]))
    row["Dataset"] = name
    comparison.append(row)

pd.DataFrame(comparison).set_index("Dataset").round(3)

In [ ]:
# Residual analysis on the validation data
best_model = xgb_models[best_group]
best_columns = feature_groups[best_group]

y_pred = best_model.predict(validation[best_columns])
residuals = y_val.to_numpy() - y_pred   # actual − predicted

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_pred, residuals, alpha=0.5, s=20)
axes[0].axhline(0, color="black", linestyle="--")
axes[0].set(title="Residuals vs Predicted", xlabel="Predicted (cm)", ylabel="Residual (cm)")

axes[1].plot(validation.index, residuals, linewidth=1)
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set(title="Residuals Over Time", xlabel="Date", ylabel="Residual (cm)")

plt.tight_layout()
plt.show()

In [ ]:
# Residuals: training vs validation
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)

for ax, (name, subset) in zip(axes, [("Training", train_fit), ("Validation", validation)]):
    predictions = best_model.predict(subset[best_columns])
    residuals = subset["target"].to_numpy() - predictions

    ax.scatter(predictions, residuals, alpha=0.35, s=15)
    ax.axhline(0, color="black", linestyle="--")
    ax.set(title=f"{name} Residuals", xlabel="Predicted (cm)")

axes[0].set_ylabel("Residual: actual − predicted (cm)")
fig.suptitle(f"XGBoost — {best_group}")
plt.tight_layout()
plt.show()

In [ ]:
# Validation error by condition (thresholds = 10th and 90th percentile of training)
low_q = train_fit["target"].quantile(0.10)
high_q = train_fit["target"].quantile(0.90)

extreme_results = hp.regime_table(
    y_val, best_model.predict(validation[best_columns]), pred_baseline,
    low_cm=low_q, high_cm=high_q
)

print(f"Low < {low_q:.0f} cm, high >= {high_q:.0f} cm")
extreme_results.round(2)

Polynomial Features

In [ ]:
# Polynomial features (degree 2) + Ridge regression for every feature group
poly_results_df, poly_models = kp.train_model_groups("Polynomial ridge", splits)

poly_results_df.round(3)

In [ ]:
# How many columns do the polynomial features create?
for group, model in poly_models.items():
    poly = model.named_steps["polynomialfeatures"]
    print(f"{group}: {len(feature_groups[group])} original → {poly.n_output_features_} polynomial features")

In [ ]:
# Compare all models on the validation data
all_results = []

for model_name, results in [("Linear regression", lr_results_df),
                            ("LightGBM", lgbm_results_df),
                            ("Random Forest", rf_results_df),
                            ("XGBoost", xgb_results_df),
                            ("Polynomial Ridge", poly_results_df)]:
    table = results[["MAE", "RMSE", "Bias", "Skill vs persistence"]].reset_index()
    table["Model"] = model_name
    all_results.append(table)

model_comparison = pd.concat(all_results).sort_values("MAE").reset_index(drop=True)

print(f"Persistence validation MAE: {persistence_mae:.3f} cm")
model_comparison.round(3)

In [ ]:
# Best model overall
best_result = model_comparison.iloc[0]

print("Best model:", best_result["Model"])
print("Best feature group:", best_result["Feature group"])
print(f"Validation MAE: {best_result['MAE']:.3f} cm")

# Error Analysis :

In [ ]:
# Low-water errors for the chosen model (training vs validation)
group = "Kaub + precipitation + temperature"
model = xgb_models[group]
columns = feature_groups[group]

tables = []

for name, subset in [("Training", train_fit), ("Validation", validation)]:
    table = hp.regime_table(subset["target"], model.predict(subset[columns]), subset["lag_0"])
    table["Dataset"] = name
    tables.append(table)

pd.concat(tables).round(2)

In [ ]:
# Validation error table
val_errors = pd.DataFrame({
    "Actual": y_val,
    "Predicted": model.predict(validation[columns]),
    "Persistence": pred_baseline
})

val_errors["Target date"] = (val_errors.index + pd.Timedelta(days=H)).date
val_errors["Error"] = val_errors["Predicted"] - val_errors["Actual"]

val_errors.head().round(2)

In [ ]:
# Forecasts vs actual levels, and the error over time
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(val_errors["Target date"], val_errors["Actual"], label="Actual", color="black")
axes[0].plot(val_errors["Target date"], val_errors["Predicted"], label="XGBoost", alpha=0.8)
axes[0].axhline(LOW_WATER_THRESHOLD, color="red", linestyle="--", label="Low-water threshold")
axes[0].set(ylabel="Water level (cm)", title="7-day-ahead forecasts against actual water levels")
axes[0].legend()

axes[1].plot(val_errors["Target date"], val_errors["Error"], color="tab:orange")
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set(ylabel="Predicted − actual (cm)", xlabel="Target date",
            title="Above zero = overprediction")

plt.tight_layout()
plt.show()

In [ ]:
# Worst low-water overpredictions (model promised more water than there was)
low_and_over = (val_errors["Actual"] < LOW_WATER_THRESHOLD) & (val_errors["Error"] > 0)

val_errors[low_and_over].nlargest(10, "Error").round(2)

Save Model for streamlit

In [ ]:
# Save the model with the pipeline (same format the Streamlit app reads)
save_group = "Kaub + precipitation + temperature"

kp.save_model(xgb_models[save_group], features=feature_groups[save_group])

print("Saved to", kp.MODEL_PATH)

In [ ]:
# Load the saved model and use it: forecast for the latest days in the data
saved = kp.load_model()

forecast = kp.predict(saved, kp.load_data())

forecast.tail()